# 11. Test on photos the model has never seen

This is the most important test. All earlier scores come from one dataset. This notebook scores **new photos**, which can be a second dataset or photos taken with your retinal adapter.

The cut-off stays fixed. **Do not change it to suit these photos**, or the result will look better than it really is.

## Prepare your photos

Make a folder inside `data/new_photos`, for example `data/new_photos/adapter_test_1`:

```
data/new_photos/adapter_test_1/
    labels.csv
    images/
        photo001.jpg
        photo002.jpg
```

`labels.csv` needs these columns:

| Column | Meaning |
|---|---|
| `filename` | The photo's file name |
| `grade` | The grade from 0 to 4, given by an eye specialist who hasn't seen the AI result |
| `patient_id` | Optional, but needed for fair confidence ranges |
| `device` | Optional, such as `iNview` or `Remidio`, to compare devices |
| `gradable` | Optional, `1` if the specialist could grade the photo, `0` if not |

**Using APTOS 2019 from Kaggle?** Put its `train.csv` in the folder, and its photos in `images`. The next cell converts its labels for you. APTOS has no patient IDs, so its confidence ranges are less reliable.

In [ ]:
%pip install -q numpy pandas pillow matplotlib scikit-learn onnxruntime

In [ ]:
from pathlib import Path

# Finds the project folder wherever this notebook is opened from.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw" / "DeepDRiD").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find the AfriEyeSamsung project folder. Open this notebook from inside it.")
PROCESSED, RESULTS, MODELS = ROOT / "data" / "processed", ROOT / "results", ROOT / "models"
print("Project folder:", ROOT)

## Settings

In [ ]:
PHOTOS = ROOT / "data" / "new_photos" / "adapter_test_1"   # change the last part to your folder's name
MODEL = "small_cnn"                                        # or the name of an exported model, such as "efficientnet_b0"

In [ ]:
import matplotlib.pyplot as plt

BLUE, TEAL, ORANGE, PURPLE, GREY = "#1F4E9E", "#00948F", "#C4581F", "#7A3E9D", "#8A94A6"
INK, MUTED, GRID = "#1B2433", "#566173", "#E3E7F0"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110, "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.edgecolor": "#AEB6C5", "axes.labelcolor": MUTED, "axes.titlecolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID, "axes.axisbelow": True,
    "legend.frameon": False,
})

In [ ]:
import json
import sys

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve

sys.path.insert(0, str(ROOT / "src"))
from afrieye.nn import Net
from afrieye.preprocess import crop_to_fundus, preprocess_pil, to_input

READY = (PHOTOS / "images").exists() and ((PHOTOS / "labels.csv").exists() or (PHOTOS / "train.csv").exists())
if not READY:
    print("No photos found yet in", PHOTOS)
    print("Follow the steps at the top of this notebook, then run it again. The cells below will skip until then.")
else:
    if not (PHOTOS / "labels.csv").exists():
        a = pd.read_csv(PHOTOS / "train.csv")
        pd.DataFrame({"filename": a.id_code + ".png", "grade": a.diagnosis}).to_csv(PHOTOS / "labels.csv", index=False)
        print("Converted APTOS labels.")
    d = pd.read_csv(PHOTOS / "labels.csv")
    d = d[d.grade.isin([0, 1, 2, 3, 4]) & d.filename.map(lambda f: (PHOTOS / "images" / f).exists())].reset_index(drop=True)
    d["grade"] = d.grade.astype(int)
    d["referable"] = (d.grade >= 2).astype(int)
    print(len(d), "labelled photos,", int(d.referable.sum()), "referable,", int((d.grade >= 3).sum()), "severe or worse")

## Score the photos

In [ ]:
def load_model(name):
    if name == "small_cnn":
        net = Net.load(MODELS / "small_cnn.npz")
        t = json.loads((MODELS / "small_cnn_thresholds.json").read_text())["thresholds"]
        def score(image):
            return net.predict(to_input(preprocess_pil(image, 192)[None]))[0]
        return score, t
    import onnxruntime as ort
    config = json.loads((MODELS / f"{name}_phone.json").read_text())
    session = ort.InferenceSession(str(MODELS / f"{name}.onnx"), providers=["CPUExecutionProvider"])
    mean, std = np.array(config["mean"], np.float32)[:, None, None], np.array(config["std"], np.float32)[:, None, None]
    def score(image):
        square = crop_to_fundus(image).resize((config["size"], config["size"]), Image.BILINEAR)
        x = np.asarray(square, np.float32).transpose(2, 0, 1) / 255
        return session.run(None, {session.get_inputs()[0].name: ((x - mean) / std)[None]})[0][0]
    return score, config["thresholds"]


if READY:
    score_photo, T = load_model(MODEL)
    P = np.stack([score_photo(Image.open(PHOTOS / "images" / f)) for f in d.filename])
    d["score"], d["urgent_score"], d["quality_score"] = P[:, 1], P[:, 2], P[:, 4]
    print("Scored", len(d), "photos with", MODEL)
    print("Cut-offs, fixed before seeing these photos:", {k: round(v, 3) for k, v in T.items()})

## The results

In [ ]:
def rates(y, s, threshold):
    tn, fp, fn, tp = confusion_matrix(y, np.asarray(s) >= threshold, labels=[0, 1]).ravel()
    return tp / max(1, tp + fn), tn / max(1, tn + fp), int(tp), int(fn), int(tn), int(fp)


def interval(frame, repeats=1000, seed=0):
    rng = np.random.default_rng(seed)
    group = frame.patient_id if "patient_id" in frame and frame.patient_id.notna().all() else pd.Series(np.arange(len(frame)), index=frame.index)
    where = list(frame.groupby(group.values).indices.values())
    y, s = frame.referable.values, frame.score.values
    out = []
    for _ in range(repeats):
        take = np.concatenate([where[i] for i in rng.integers(0, len(where), len(where))])
        if 0 < y[take].sum() < len(take):
            out.append(roc_auc_score(y[take], s[take]))
    return float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))


if READY:
    if d.referable.nunique() < 2:
        raise ValueError("The test needs both referable and non-referable photos.")
    sens, spec, tp, fn, tn, fp = rates(d.referable, d.score, T["referable"])
    low, high = interval(d)
    result = {
        "photos": PHOTOS.name, "model": MODEL, "n": len(d), "referable": int(d.referable.sum()), "auc": float(roc_auc_score(d.referable, d.score)), "auc_ci": [low, high],
        "sensitivity": float(sens), "specificity": float(spec), "tp": tp, "fn": fn, "tn": tn, "fp": fp,
        "severe_missed": int(((d.grade >= 3) & (d.score < T["referable"])).sum()), "severe_total": int((d.grade >= 3).sum()),
        "asked_to_retake": float((d.quality_score < T["quality"]).mean()),
    }
    print(f"AUC {result['auc']:.3f} (95% range {low:.2f} to {high:.2f})")
    print(f"Sensitivity {sens:.1%} ({tp} of {tp + fn}) | Specificity {spec:.1%} ({tn} of {tn + fp})")
    print(f"Severe or worse missed: {result['severe_missed']} of {result['severe_total']}")
    print(f"Quality score below the retake cut-off: {result['asked_to_retake']:.1%}")
    if "gradable" in d:
        print(f"Photos the specialist could grade: {d.gradable.mean():.1%}")
    if "device" in d:
        rows = []
        for device, g in d.groupby("device"):
            if g.referable.nunique() == 2:
                a, b, *_ = rates(g.referable, g.score, T["referable"])
                rows.append({"Device": device, "Photos": len(g), "AUC": round(roc_auc_score(g.referable, g.score), 3), "Sensitivity": f"{a:.1%}", "Specificity": f"{b:.1%}"})
        print(pd.DataFrame(rows).to_string(index=False))
    fpr, tpr, _ = roc_curve(d.referable, d.score)
    fig, ax = plt.subplots(figsize=(5.4, 4.6))
    ax.plot([0, 1], [0, 1], color="#AEB6C5", linestyle="--", linewidth=1, label="Chance")
    ax.plot(fpr, tpr, color=TEAL, linewidth=2, label=f"{MODEL}, AUC {result['auc']:.2f}")
    ax.plot(1 - spec, sens, "o", color=TEAL, markersize=9, markeredgecolor="white", label="Fixed cut-off")
    ax.set(xlabel="False positive rate", ylabel="Sensitivity", xlim=(0, 1), ylim=(0, 1.02), title=f"New photos: {PHOTOS.name}"); ax.legend(loc="lower right", fontsize=9)
    fig.tight_layout(); plt.show()
    (RESULTS / f"new_photos_{PHOTOS.name}_{MODEL}.json").write_text(json.dumps(result, indent=1))
    d.drop(columns=[c for c in ("gradable",) if c in d]).to_csv(RESULTS / f"new_photos_{PHOTOS.name}_{MODEL}.csv", index=False)
    print("Saved to results")